# 04 - SQL Business Analysis

## Objective

The objective of this notebook is to validate and deepen the insights found during the Exploratory Data Analysis phase using SQL.

This phase focuses on answering business questions with structured SQL queries.

The goal is to demonstrate practical SQL skills such as:

- `JOIN`
- `GROUP BY`
- `CASE WHEN`
- Common Table Expressions (`CTE`)
- Window Functions
- Ranking
- Running totals
- Year-over-year growth
- Pareto analysis

This notebook uses the cleaned dataset generated in `02_data_cleaning.ipynb`.

## Business Problem Reminder

Nova Retail has strong sales, but profitability does not always grow at the same pace.

The main question remains:

> **How can Nova Retail improve profitability without simply increasing sales?**

## How to Read This Notebook

Each section follows the same logic:

1. **Business question**: what decision are we trying to support?
2. **SQL query**: how do we answer the question with SQL?
3. **Business interpretation**: what should we look for in the result?

This notebook is designed to be readable by both technical and non-technical audiences.

## 1. Notebook Setup

This section imports the required libraries and defines helper functions for running SQL queries.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


def format_currency(value):
    if pd.isna(value):
        return "N/A"
    return f"${value:,.0f}"


def format_percent(value):
    if pd.isna(value):
        return "N/A"
    return f"{value:.2%}"


def run_query(query, connection, title=None):
    """Run a SQL query and display the result as a DataFrame."""
    if title:
        print(title)
    result = pd.read_sql_query(query, connection)
    display(result)
    return result

## 2. Load the Cleaned Dataset

The SQL phase should use the cleaned dataset:

```text
data/cleaned/cleaned_superstore.csv
```

If this file is not found, the notebook can fall back to the raw dataset for demonstration purposes and apply the same cleaning logic.

In [ ]:
possible_cleaned_paths = [
    Path("../data/cleaned/cleaned_superstore.csv"),
    Path("data/cleaned/cleaned_superstore.csv"),
    Path("cleaned_superstore.csv")
]

possible_raw_paths = [
    Path("../data/raw/superstore.csv"),
    Path("data/raw/superstore.csv"),
    Path("superstore.csv")
]

DATA_PATH = None
DATA_SOURCE = None

for path in possible_cleaned_paths:
    if path.exists():
        DATA_PATH = path
        DATA_SOURCE = "cleaned"
        break

if DATA_PATH is None:
    for path in possible_raw_paths:
        if path.exists():
            DATA_PATH = path
            DATA_SOURCE = "raw"
            break

if DATA_PATH is None:
    raise FileNotFoundError(
        "No dataset was found. Please make sure cleaned_superstore.csv or superstore.csv is available."
    )

print(f"Dataset loaded from: {DATA_PATH}")
print(f"Data source type: {DATA_SOURCE}")

df = pd.read_csv(DATA_PATH)
display(df.head())

## 3. Prepare Dataset for SQL

If the cleaned dataset is already loaded, this section only checks that required columns exist.

If the raw dataset is loaded, the necessary cleaning steps are applied so that the SQL analysis can still run.

In [ ]:
def prepare_superstore_data(data, source_type):
    data = data.copy()

    if source_type == "raw":
        rename_columns = {
            "Category": "category",
            "City": "city",
            "Country": "country",
            "Customer.ID": "customer_id",
            "Customer.Name": "customer_name",
            "Discount": "discount",
            "Market": "market",
            "记录数": "record_count",
            "Order.Date": "order_date",
            "Order.ID": "order_id",
            "Order.Priority": "order_priority",
            "Product.ID": "product_id",
            "Product.Name": "product_name",
            "Profit": "profit",
            "Quantity": "quantity",
            "Region": "region",
            "Row.ID": "row_id",
            "Sales": "sales",
            "Segment": "segment",
            "Ship.Date": "ship_date",
            "Ship.Mode": "ship_mode",
            "Shipping.Cost": "shipping_cost",
            "State": "state",
            "Sub.Category": "sub_category",
            "Year": "order_year",
            "Market2": "market_group",
            "weeknum": "order_week"
        }
        data = data.rename(columns=rename_columns)

        if "record_count" in data.columns:
            data = data.drop(columns=["record_count"])

    # Clean text values
    text_columns = data.select_dtypes(include="object").columns
    for col in text_columns:
        data[col] = data[col].astype(str).str.strip()

    # Convert dates
    data["order_date"] = pd.to_datetime(data["order_date"], errors="coerce")
    data["ship_date"] = pd.to_datetime(data["ship_date"], errors="coerce")

    # Create variables if missing
    if "order_year" not in data.columns:
        data["order_year"] = data["order_date"].dt.year

    if "order_month" not in data.columns:
        data["order_month"] = data["order_date"].dt.month

    if "order_month_name" not in data.columns:
        data["order_month_name"] = data["order_date"].dt.month_name()

    if "order_quarter" not in data.columns:
        data["order_quarter"] = "Q" + data["order_date"].dt.quarter.astype(str)

    if "order_year_month" not in data.columns:
        data["order_year_month"] = data["order_date"].dt.to_period("M").astype(str)

    if "shipping_delay_days" not in data.columns:
        data["shipping_delay_days"] = (data["ship_date"] - data["order_date"]).dt.days

    if "profit_margin" not in data.columns:
        data["profit_margin"] = np.where(
            data["sales"] != 0,
            data["profit"] / data["sales"],
            np.nan
        )

    if "profit_status" not in data.columns:
        data["profit_status"] = np.select(
            [data["profit"] > 0, data["profit"] < 0, data["profit"] == 0],
            ["Profitable", "Loss-making", "Break-even"],
            default="Unknown"
        )

    if "discount_band" not in data.columns:
        conditions = [
            data["discount"] == 0,
            (data["discount"] > 0) & (data["discount"] <= 0.10),
            (data["discount"] > 0.10) & (data["discount"] <= 0.20),
            (data["discount"] > 0.20) & (data["discount"] <= 0.30),
            (data["discount"] > 0.30) & (data["discount"] <= 0.50),
            data["discount"] > 0.50
        ]
        choices = [
            "No discount",
            "Low discount",
            "Moderate discount",
            "High discount",
            "Very high discount",
            "Extreme discount"
        ]
        data["discount_band"] = np.select(conditions, choices, default="Unknown")

    if "shipping_cost_ratio" not in data.columns:
        data["shipping_cost_ratio"] = np.where(
            data["sales"] != 0,
            data["shipping_cost"] / data["sales"],
            np.nan
        )

    if "order_size" not in data.columns:
        data["order_size"] = pd.cut(
            data["quantity"],
            bins=[0, 2, 5, 10, np.inf],
            labels=["Small", "Medium", "Large", "Very Large"],
            include_lowest=True
        ).astype(str)

    if "market_group" not in data.columns:
        data["market_group"] = data.get("market", "Unknown")

    # Convert date columns to strings for SQLite compatibility
    data["order_date"] = data["order_date"].dt.strftime("%Y-%m-%d")
    data["ship_date"] = data["ship_date"].dt.strftime("%Y-%m-%d")

    return data


df = prepare_superstore_data(df, DATA_SOURCE)

required_columns = [
    "row_id", "order_id", "order_date", "ship_date", "customer_id", "customer_name",
    "segment", "product_id", "product_name", "category", "sub_category",
    "city", "state", "country", "region", "market", "sales", "profit",
    "quantity", "discount", "shipping_cost", "ship_mode", "order_priority",
    "order_year", "order_month", "order_quarter", "order_year_month",
    "shipping_delay_days", "profit_margin", "profit_status", "discount_band",
    "shipping_cost_ratio", "order_size"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print("Dataset is ready for SQL modeling.")
display(df.head())

## 4. Build a Simple Relational Model

The cleaned dataset is originally a single table.

To practice SQL in a more realistic way, it will be transformed into a simple star-schema style model:

- `fact_sales`
- `dim_customers`
- `dim_products`
- `dim_geography`

This allows the analysis to use `JOIN` operations instead of querying only one flat table.

### Product Key Note

`product_id` is not treated as the only product identifier.

Some product IDs may be associated with multiple product names or descriptions, so a new `product_key` is created using the combination of:

```text
product_id + product_name + category + sub_category
```

In [ ]:
df_model = df.copy()

# Create robust keys for dimensions
df_model["product_key"] = (
    df_model["product_id"].astype(str) + " | " +
    df_model["product_name"].astype(str) + " | " +
    df_model["category"].astype(str) + " | " +
    df_model["sub_category"].astype(str)
)

df_model["geography_key"] = (
    df_model["market"].astype(str) + " | " +
    df_model["region"].astype(str) + " | " +
    df_model["country"].astype(str) + " | " +
    df_model["state"].astype(str) + " | " +
    df_model["city"].astype(str)
)

dim_customers = (
    df_model[["customer_id", "customer_name", "segment"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_products = (
    df_model[["product_key", "product_id", "product_name", "category", "sub_category"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_geography = (
    df_model[["geography_key", "market", "market_group", "region", "country", "state", "city"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

fact_sales_columns = [
    "row_id", "order_id", "order_date", "ship_date", "customer_id", "product_key",
    "geography_key", "sales", "profit", "quantity", "discount", "shipping_cost",
    "ship_mode", "order_priority", "order_year", "order_month", "order_month_name",
    "order_quarter", "order_year_month", "order_week", "shipping_delay_days",
    "profit_margin", "profit_status", "discount_band", "shipping_cost_ratio",
    "order_size"
]

# Keep only fact columns that exist in the dataset
fact_sales_columns = [col for col in fact_sales_columns if col in df_model.columns]

fact_sales = df_model[fact_sales_columns].copy()

model_summary = pd.DataFrame({
    "Table": ["fact_sales", "dim_customers", "dim_products", "dim_geography"],
    "Rows": [
        fact_sales.shape[0],
        dim_customers.shape[0],
        dim_products.shape[0],
        dim_geography.shape[0]
    ],
    "Columns": [
        fact_sales.shape[1],
        dim_customers.shape[1],
        dim_products.shape[1],
        dim_geography.shape[1]
    ]
})

model_summary

## 5. Load Tables into SQLite

SQLite is used here because it runs easily in a notebook without external setup.

The same SQL logic can later be adapted to PostgreSQL, MySQL, or SQL Server.

In [ ]:
possible_db_dirs = [
    Path("../sql"),
    Path("sql"),
    Path(".")
]

DB_DIR = possible_db_dirs[0]

if not DB_DIR.exists():
    DB_DIR = possible_db_dirs[1]

DB_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "nova_retail.db"

connection = sqlite3.connect(DB_PATH)

fact_sales.to_sql("fact_sales", connection, if_exists="replace", index=False)
dim_customers.to_sql("dim_customers", connection, if_exists="replace", index=False)
dim_products.to_sql("dim_products", connection, if_exists="replace", index=False)
dim_geography.to_sql("dim_geography", connection, if_exists="replace", index=False)

print(f"SQLite database created at: {DB_PATH}")

In [ ]:
tables_check = run_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection,
    title="Tables available in the SQLite database:"
)

## 6. SQL Analysis Questions

The following SQL queries validate and deepen the EDA findings.

The queries are grouped by business area.

## 6.1 Overall KPI Summary

### Business Question

**Is Nova Retail profitable overall?**

In [ ]:
query_kpi_summary = """
SELECT
    COUNT(*) AS order_lines,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_rows,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_row_share
FROM fact_sales;
"""

kpi_sql = run_query(query_kpi_summary, connection, "KPI Summary")

### Interpretation

This query checks whether the company is profitable overall and how many transactions are loss-making.

A high share of loss-making rows means that overall profit may be hiding important business issues.

## 6.2 Yearly Sales and Profit Growth

### Business Question

**Are sales and profit growing at the same pace?**

In [ ]:
query_yearly_growth = """
WITH yearly_performance AS (
    SELECT
        order_year,
        ROUND(SUM(sales), 2) AS sales,
        ROUND(SUM(profit), 2) AS profit,
        ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin
    FROM fact_sales
    GROUP BY order_year
),
yearly_with_lag AS (
    SELECT
        order_year,
        sales,
        profit,
        profit_margin,
        LAG(sales) OVER (ORDER BY order_year) AS previous_year_sales,
        LAG(profit) OVER (ORDER BY order_year) AS previous_year_profit
    FROM yearly_performance
)
SELECT
    order_year,
    sales,
    profit,
    profit_margin,
    ROUND((sales - previous_year_sales) / NULLIF(previous_year_sales, 0), 4) AS sales_growth,
    ROUND((profit - previous_year_profit) / NULLIF(previous_year_profit, 0), 4) AS profit_growth
FROM yearly_with_lag
ORDER BY order_year;
"""

yearly_growth_sql = run_query(query_yearly_growth, connection, "Yearly Sales and Profit Growth")

### Interpretation

This query uses a CTE and `LAG()` window function to compare each year with the previous year.

If sales growth is higher than profit growth, the business may be growing in volume but not in efficiency.

## 6.3 Monthly Running Totals

### Business Question

**How do cumulative sales and cumulative profit evolve over time?**

In [ ]:
query_monthly_running_total = """
WITH monthly_performance AS (
    SELECT
        order_year_month,
        ROUND(SUM(sales), 2) AS sales,
        ROUND(SUM(profit), 2) AS profit
    FROM fact_sales
    GROUP BY order_year_month
)
SELECT
    order_year_month,
    sales,
    profit,
    ROUND(SUM(sales) OVER (ORDER BY order_year_month), 2) AS running_sales,
    ROUND(SUM(profit) OVER (ORDER BY order_year_month), 2) AS running_profit
FROM monthly_performance
ORDER BY order_year_month;
"""

monthly_running_total_sql = run_query(query_monthly_running_total, connection, "Monthly Running Totals")

### Interpretation

Running totals help show whether profitability accumulates consistently over time or whether certain periods reduce the overall trajectory.

## 6.4 Category and Sub-Category Profitability

### Business Question

**Which categories and sub-categories create or destroy business value?**

In [ ]:
query_category_performance = """
SELECT
    p.category,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    SUM(f.quantity) AS quantity,
    COUNT(DISTINCT f.order_id) AS orders,
    SUM(CASE WHEN f.profit < 0 THEN 1 ELSE 0 END) AS loss_making_rows
FROM fact_sales f
JOIN dim_products p
    ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY profit DESC;
"""

category_performance_sql = run_query(query_category_performance, connection, "Category Performance")

In [ ]:
query_sub_category_performance = """
SELECT
    p.category,
    p.sub_category,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    SUM(f.quantity) AS quantity,
    COUNT(DISTINCT f.order_id) AS orders,
    SUM(CASE WHEN f.profit < 0 THEN 1 ELSE 0 END) AS loss_making_rows,
    RANK() OVER (ORDER BY SUM(f.profit) DESC) AS profit_rank
FROM fact_sales f
JOIN dim_products p
    ON f.product_key = p.product_key
GROUP BY p.category, p.sub_category
ORDER BY profit ASC;
"""

sub_category_performance_sql = run_query(query_sub_category_performance, connection, "Sub-Category Performance")

### Interpretation

This analysis helps identify whether revenue and profit are aligned.

A sub-category with high sales but negative profit should be investigated further.

## 6.5 Product Ranking

### Business Question

**Which products are the strongest and weakest contributors to profit?**

In [ ]:
query_top_products_by_profit = """
WITH product_performance AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        ROUND(SUM(f.sales), 2) AS sales,
        ROUND(SUM(f.profit), 2) AS profit,
        ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
        COUNT(DISTINCT f.order_id) AS orders
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
),
ranked_products AS (
    SELECT
        *,
        RANK() OVER (ORDER BY profit DESC) AS profit_rank,
        RANK() OVER (ORDER BY sales DESC) AS sales_rank
    FROM product_performance
)
SELECT
    product_id,
    product_name,
    category,
    sub_category,
    sales,
    profit,
    profit_margin,
    orders,
    profit_rank,
    sales_rank
FROM ranked_products
WHERE profit_rank <= 10
ORDER BY profit_rank;
"""

top_products_by_profit_sql = run_query(query_top_products_by_profit, connection, "Top 10 Products by Profit")

In [ ]:
query_loss_making_products = """
WITH product_performance AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        ROUND(SUM(f.sales), 2) AS sales,
        ROUND(SUM(f.profit), 2) AS profit,
        ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
        COUNT(DISTINCT f.order_id) AS orders
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
)
SELECT
    product_id,
    product_name,
    category,
    sub_category,
    sales,
    profit,
    profit_margin,
    orders
FROM product_performance
ORDER BY profit ASC
LIMIT 20;
"""

loss_making_products_sql = run_query(query_loss_making_products, connection, "Top 20 Loss-Making Products")

### Interpretation

The most profitable products and the best-selling products are not always the same.

This is why profitability should be evaluated with both sales and margin.

## 6.6 High Sales but Low Margin Products

### Business Question

**Which products sell well but do not generate enough profit?**

In [ ]:
query_high_sales_low_margin_products = """
WITH product_performance AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        SUM(f.sales) AS sales,
        SUM(f.profit) AS profit,
        SUM(f.profit) / NULLIF(SUM(f.sales), 0) AS profit_margin,
        COUNT(DISTINCT f.order_id) AS orders
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
),
overall_margin AS (
    SELECT
        SUM(profit) / NULLIF(SUM(sales), 0) AS company_profit_margin
    FROM fact_sales
),
average_product_sales AS (
    SELECT
        AVG(sales) AS avg_product_sales
    FROM product_performance
)
SELECT
    pp.product_id,
    pp.product_name,
    pp.category,
    pp.sub_category,
    ROUND(pp.sales, 2) AS sales,
    ROUND(pp.profit, 2) AS profit,
    ROUND(pp.profit_margin, 4) AS profit_margin,
    pp.orders
FROM product_performance pp
CROSS JOIN overall_margin om
CROSS JOIN average_product_sales aps
WHERE pp.sales > aps.avg_product_sales
  AND pp.profit_margin < om.company_profit_margin
ORDER BY pp.sales DESC
LIMIT 20;
"""

high_sales_low_margin_products_sql = run_query(
    query_high_sales_low_margin_products,
    connection,
    "High Sales but Low Margin Products"
)

### Interpretation

These products may look successful because they generate high revenue.

However, they may reduce business efficiency if their margins are below the company average.

## 6.7 Customer Segment Performance

### Business Question

**Which customer segments generate the most value?**

In [ ]:
query_segment_performance = """
SELECT
    c.segment,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    COUNT(DISTINCT f.order_id) AS orders,
    COUNT(DISTINCT f.customer_id) AS customers
FROM fact_sales f
JOIN dim_customers c
    ON f.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY profit DESC;
"""

segment_performance_sql = run_query(query_segment_performance, connection, "Customer Segment Performance")

In [ ]:
query_top_customers = """
WITH customer_performance AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.segment,
        ROUND(SUM(f.sales), 2) AS sales,
        ROUND(SUM(f.profit), 2) AS profit,
        ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
        COUNT(DISTINCT f.order_id) AS orders
    FROM fact_sales f
    JOIN dim_customers c
        ON f.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name, c.segment
)
SELECT
    *,
    RANK() OVER (ORDER BY profit DESC) AS profit_rank
FROM customer_performance
ORDER BY profit DESC
LIMIT 20;
"""

top_customers_sql = run_query(query_top_customers, connection, "Top 20 Customers by Profit")

### Interpretation

A customer should not be evaluated only by revenue.

A customer with high sales but low margin may require a different commercial strategy.

## 6.8 Geographic Performance

### Business Question

**Which markets and countries are the most and least profitable?**

In [ ]:
query_market_performance = """
SELECT
    g.market,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    COUNT(DISTINCT f.order_id) AS orders,
    COUNT(DISTINCT f.customer_id) AS customers
FROM fact_sales f
JOIN dim_geography g
    ON f.geography_key = g.geography_key
GROUP BY g.market
ORDER BY profit DESC;
"""

market_performance_sql = run_query(query_market_performance, connection, "Market Performance")

In [ ]:
query_country_performance = """
SELECT
    g.market,
    g.region,
    g.country,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    COUNT(DISTINCT f.order_id) AS orders
FROM fact_sales f
JOIN dim_geography g
    ON f.geography_key = g.geography_key
GROUP BY g.market, g.region, g.country
ORDER BY profit ASC
LIMIT 20;
"""

country_performance_sql = run_query(country_performance_sql if False else query_country_performance, connection, "Bottom 20 Countries by Profit")

### Interpretation

Markets with high revenue but weak margin should be reviewed.

The goal is to identify where growth is profitable and where growth may hide losses.

## 6.9 Discount Impact Analysis

### Business Question

**Do discounts improve or reduce profitability?**

In [ ]:
query_discount_impact = """
WITH discount_analysis AS (
    SELECT
        CASE
            WHEN discount = 0 THEN 'No discount'
            WHEN discount > 0 AND discount <= 0.10 THEN 'Low discount'
            WHEN discount > 0.10 AND discount <= 0.20 THEN 'Moderate discount'
            WHEN discount > 0.20 AND discount <= 0.30 THEN 'High discount'
            WHEN discount > 0.30 AND discount <= 0.50 THEN 'Very high discount'
            WHEN discount > 0.50 THEN 'Extreme discount'
            ELSE 'Unknown'
        END AS discount_band,
        CASE
            WHEN discount = 0 THEN 1
            WHEN discount > 0 AND discount <= 0.10 THEN 2
            WHEN discount > 0.10 AND discount <= 0.20 THEN 3
            WHEN discount > 0.20 AND discount <= 0.30 THEN 4
            WHEN discount > 0.30 AND discount <= 0.50 THEN 5
            WHEN discount > 0.50 THEN 6
            ELSE 7
        END AS discount_band_order,
        sales,
        profit,
        order_id
    FROM fact_sales
)
SELECT
    discount_band,
    ROUND(SUM(sales), 2) AS sales,
    ROUND(SUM(profit), 2) AS profit,
    ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin,
    COUNT(*) AS order_lines,
    COUNT(DISTINCT order_id) AS orders,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_rows,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_rate
FROM discount_analysis
GROUP BY discount_band, discount_band_order
ORDER BY discount_band_order;
"""

discount_impact_sql = run_query(query_discount_impact, connection, "Discount Impact Analysis")

### Interpretation

This query uses `CASE WHEN` to classify discount levels.

If higher discounts have lower margins or higher loss rates, the discount strategy should be reviewed.

## 6.10 Shipping and Operational Performance

### Business Question

**Do shipping methods and order priorities affect profitability?**

In [ ]:
query_shipping_performance = """
SELECT
    ship_mode,
    ROUND(SUM(sales), 2) AS sales,
    ROUND(SUM(profit), 2) AS profit,
    ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin,
    ROUND(SUM(shipping_cost), 2) AS shipping_cost,
    ROUND(SUM(shipping_cost) / NULLIF(SUM(sales), 0), 4) AS shipping_cost_ratio,
    ROUND(AVG(shipping_delay_days), 2) AS avg_shipping_delay_days,
    COUNT(DISTINCT order_id) AS orders
FROM fact_sales
GROUP BY ship_mode
ORDER BY profit DESC;
"""

shipping_performance_sql = run_query(query_shipping_performance, connection, "Shipping Mode Performance")

In [ ]:
query_priority_performance = """
SELECT
    order_priority,
    ROUND(SUM(sales), 2) AS sales,
    ROUND(SUM(profit), 2) AS profit,
    ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin,
    ROUND(SUM(shipping_cost), 2) AS shipping_cost,
    COUNT(DISTINCT order_id) AS orders
FROM fact_sales
GROUP BY order_priority
ORDER BY profit DESC;
"""

priority_performance_sql = run_query(query_priority_performance, connection, "Order Priority Performance")

### Interpretation

Shipping should be evaluated using both cost and profitability.

A shipping method is not necessarily bad because it costs more. It becomes a problem if the additional cost does not support profitable sales.

## 6.11 Pareto Analysis

### Business Question

**How concentrated is profit among products?**

This analysis checks how many profitable products are needed to generate around 80% of total positive profit.

In [ ]:
query_pareto_products = """
WITH product_profit AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        SUM(f.profit) AS profit
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
    HAVING SUM(f.profit) > 0
),
ranked_products AS (
    SELECT
        *,
        ROW_NUMBER() OVER (ORDER BY profit DESC) AS product_rank,
        COUNT(*) OVER () AS total_profitable_products,
        SUM(profit) OVER (ORDER BY profit DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_profit,
        SUM(profit) OVER () AS total_positive_profit
    FROM product_profit
),
pareto AS (
    SELECT
        product_id,
        product_name,
        category,
        sub_category,
        ROUND(profit, 2) AS profit,
        product_rank,
        total_profitable_products,
        ROUND(cumulative_profit, 2) AS cumulative_profit,
        ROUND(cumulative_profit / NULLIF(total_positive_profit, 0), 4) AS cumulative_profit_share,
        ROUND(1.0 * product_rank / total_profitable_products, 4) AS product_share
    FROM ranked_products
)
SELECT *
FROM pareto
WHERE cumulative_profit_share <= 0.80
ORDER BY product_rank;
"""

pareto_products_sql = run_query(query_pareto_products, connection, "Products Contributing to Around 80% of Positive Profit")

In [ ]:
query_pareto_summary = """
WITH product_profit AS (
    SELECT
        p.product_id,
        p.product_name,
        SUM(f.profit) AS profit
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name
    HAVING SUM(f.profit) > 0
),
ranked_products AS (
    SELECT
        *,
        ROW_NUMBER() OVER (ORDER BY profit DESC) AS product_rank,
        COUNT(*) OVER () AS total_profitable_products,
        SUM(profit) OVER (ORDER BY profit DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_profit,
        SUM(profit) OVER () AS total_positive_profit
    FROM product_profit
),
first_rank_reaching_80pct AS (
    SELECT
        MIN(product_rank) AS products_needed_for_80pct_profit
    FROM ranked_products
    WHERE cumulative_profit / NULLIF(total_positive_profit, 0) >= 0.80
)
SELECT
    products_needed_for_80pct_profit,
    total_profitable_products,
    ROUND(1.0 * products_needed_for_80pct_profit / total_profitable_products, 4) AS share_of_profitable_products
FROM first_rank_reaching_80pct
CROSS JOIN (
    SELECT COUNT(*) AS total_profitable_products FROM product_profit
);
"""

pareto_summary_sql = run_query(query_pareto_summary, connection, "Pareto Summary")

### Interpretation

If a small share of products generates most of the profit, the company should protect and prioritize those products.

This can guide product strategy, promotion decisions, and inventory focus.

## 7. SQL Key Findings

This section summarizes the main findings from the SQL analysis.

The values below are generated from the SQL query outputs.

In [ ]:
# Extract a few values from SQL output tables for a simple findings summary
overall_margin = kpi_sql.loc[0, "profit_margin"]
loss_share = kpi_sql.loc[0, "loss_making_row_share"]

top_category = category_performance_sql.sort_values("profit", ascending=False).iloc[0]
lowest_sub_category = sub_category_performance_sql.sort_values("profit", ascending=True).iloc[0]

top_market = market_performance_sql.sort_values("profit", ascending=False).iloc[0]
lowest_market = market_performance_sql.sort_values("profit", ascending=True).iloc[0]

best_discount_band = discount_impact_sql.sort_values("profit_margin", ascending=False).iloc[0]
worst_discount_band = discount_impact_sql.sort_values("profit_margin", ascending=True).iloc[0]

sql_key_findings = pd.DataFrame({
    "Business Area": [
        "Overall Profitability",
        "Loss-Making Transactions",
        "Product Category",
        "Sub-Category Risk",
        "Market Performance",
        "Market Performance",
        "Discount Strategy",
        "Discount Strategy"
    ],
    "SQL Finding": [
        f"Overall profit margin is {overall_margin:.2%}.",
        f"{loss_share:.2%} of order lines are loss-making.",
        f"The most profitable category is {top_category['category']}.",
        f"The weakest sub-category by profit is {lowest_sub_category['sub_category']}.",
        f"The most profitable market is {top_market['market']}.",
        f"The weakest market by profit is {lowest_market['market']}.",
        f"The best discount band by margin is {best_discount_band['discount_band']}.",
        f"The worst discount band by margin is {worst_discount_band['discount_band']}."
    ]
})

sql_key_findings

## 8. Export SQL Outputs

The main SQL query results are exported for documentation, reporting, and dashboard preparation.

In [ ]:
possible_output_dirs = [
    Path("../reports/sql_outputs"),
    Path("reports/sql_outputs"),
    Path("sql_outputs")
]

OUTPUT_DIR = possible_output_dirs[0]

if not OUTPUT_DIR.parent.exists():
    OUTPUT_DIR = possible_output_dirs[1]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sql_exports = {
    "sql_kpi_summary.csv": kpi_sql,
    "sql_yearly_growth.csv": yearly_growth_sql,
    "sql_monthly_running_total.csv": monthly_running_total_sql,
    "sql_category_performance.csv": category_performance_sql,
    "sql_sub_category_performance.csv": sub_category_performance_sql,
    "sql_top_products_by_profit.csv": top_products_by_profit_sql,
    "sql_loss_making_products.csv": loss_making_products_sql,
    "sql_high_sales_low_margin_products.csv": high_sales_low_margin_products_sql,
    "sql_segment_performance.csv": segment_performance_sql,
    "sql_top_customers.csv": top_customers_sql,
    "sql_market_performance.csv": market_performance_sql,
    "sql_country_performance.csv": country_performance_sql,
    "sql_discount_impact.csv": discount_impact_sql,
    "sql_shipping_performance.csv": shipping_performance_sql,
    "sql_priority_performance.csv": priority_performance_sql,
    "sql_pareto_products.csv": pareto_products_sql,
    "sql_pareto_summary.csv": pareto_summary_sql,
    "sql_key_findings.csv": sql_key_findings
}

for filename, table in sql_exports.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

print(f"SQL output tables exported to: {OUTPUT_DIR}")

## 9. Conclusion

The SQL analysis validates and deepens the insights found during the EDA phase.

This phase demonstrated how SQL can be used to answer business questions about:

- overall profitability
- sales and profit growth
- product performance
- loss-making products
- customer segments
- geographic performance
- discount impact
- shipping performance
- Pareto concentration

The next step is to use these findings to design the dashboard and final business recommendations.

## Next Phase

```text
05_dashboard_planning
```

The dashboard phase will transform the most important insights into clear visuals for decision-makers.